# Post-Quantum MQTT/TLS 1.3 Performance Evaluation

**Research project:** ML-KEM-768 vs classical TLS baselines under controlled network conditions

**Author:** Mehboob Ahmad

This notebook is designed to be uploaded to GitHub and run primarily in **Google Colab**. It also contains commands and notes for running the software locally on a **MacBook Pro**.

## Research goal

We reproduce the core experimental idea of the paper:

> *Network Conditions Dominate MQTT-over-TLS 1.3 Performance: A Comparative Evaluation of ML-KEM-768, RSA-2048, and ECC-P256*

The paper evaluates MQTT-over-TLS 1.3 across latency and packet-loss conditions and measures handshake time, CPU time, memory, transmitted bytes, and connection reliability.

### Important methodological note

A normal Python `ssl` installation does **not** automatically make ML-KEM-768 a TLS 1.3 key-establishment group. ML-KEM support must come from an OpenSSL build/provider that exposes PQ key exchange to TLS 1.3. Open Quantum Safe's `oqs-provider` is specifically designed for this purpose.

Therefore this notebook is deliberately split into:

1. **Classical MQTT/TLS 1.3 baseline** — fully reproducible in the notebook.
2. **ML-KEM-768 cryptographic validation/benchmark** — validates the actual ML-KEM primitive.
3. **PQC TLS integration** — optional advanced stage using OpenSSL 3 + OQS provider.
4. **Network impairment and experiment automation**.
5. **Statistical analysis and plots**.

Do not claim that the ML-KEM results are TLS-level results until the OQS/OpenSSL integration section successfully negotiates an ML-KEM TLS group.


## 0. Environment strategy

### Google Colab

Use Colab for the complete Linux-based experiment because it provides a Linux environment where tools such as `tc/netem` can be used.

### MacBook Pro

Your MacBook can run:

- Python experiments
- Mosquitto
- OpenSSL 3
- liboqs / ML-KEM
- OQS provider
- classical TLS tests

However, Linux `tc/netem` is not available natively on macOS. For packet-loss/latency experiments matching the paper's Linux approach, use **Google Colab or a Linux VM/container**.

The notebook detects the platform so that commands can fail gracefully instead of silently producing invalid measurements.


In [1]:
import os
import sys
import platform
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("Working directory:", Path.cwd())

if "google.colab" in sys.modules:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"

print("Detected environment:", ENVIRONMENT)


Python: 3.14.6 (v3.14.6:c63aec69bd5, Jun 10 2026, 08:07:54) [Clang 21.0.0 (clang-2100.1.1.101)]
Platform: macOS-12.7.6-x86_64-i386-64bit-Mach-O
Machine: x86_64
Working directory: /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT
Detected environment: local


**Expected result:** You should see your Python version, operating system, CPU architecture, and either `colab` or `local`.

**Why this cell matters:** Reproducible experiments must record the machine/environment because CPU, operating-system scheduling, OpenSSL version, and Python version can affect timing measurements.


## 1. Install Python packages

These packages handle MQTT communication, cryptography, process measurements, data analysis, and plotting.


In [2]:
%pip install -q paho-mqtt psutil pandas numpy matplotlib seaborn scipy statsmodels cryptography
print("Python dependencies installed.")



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Python dependencies installed.


**Expected result:** `Python dependencies installed.`

`paho-mqtt` is the MQTT client, `psutil` measures process CPU/RSS, `pandas` stores results, and `scipy/statsmodels` support statistical analysis.

For a GitHub repository, keep a matching `requirements.txt` file so another researcher can recreate the Python environment.


## 2. Check OpenSSL

The paper used OpenSSL 3.x. For PQC TLS experiments, OpenSSL 3 is required by the OQS provider.


In [3]:
import shutil
import subprocess

openssl = shutil.which("openssl")

if openssl:
    result = subprocess.run([openssl, "version", "-a"], capture_output=True, text=True)
    print(result.stdout)
else:
    print("OpenSSL executable was not found.")


OpenSSL 3.6.3 9 Jun 2026 (Library: OpenSSL 3.6.3 9 Jun 2026)
built on: Tue Jun  9 11:46:57 2026 UTC
platform: darwin64-x86_64-cc
options:  bn(64,64)
compiler: clang -fPIC -arch x86_64 -O3 -Wall -DL_ENDIAN -DOPENSSL_PIC -D_REENTRANT -DOPENSSL_BUILDING_OPENSSL -DNDEBUG
OPENSSLDIR: "/usr/local/etc/openssl@3"
ENGINESDIR: "/usr/local/Cellar/openssl@3/3.6.3/lib/engines-3"
MODULESDIR: "/usr/local/Cellar/openssl@3/3.6.3/lib/ossl-modules"
Seeding source: os-specific
CPUINFO: OPENSSL_ia32cap=0x7ffaf3ffffebffff:0x00000000029c6fbf:0x00000000bc002e00:0x0000000000000000:0x0000000000000000



**Expected result:** On a modern Linux/Colab system you should see `OpenSSL 3.x`.

On macOS, the system-provided `/usr/bin/openssl` may be LibreSSL or an older OpenSSL. For the advanced PQC stage, install OpenSSL 3 with Homebrew and make sure the intended binary is first on your `PATH`.


## 3. MacBook-only setup

Run the following commands in the **Mac Terminal**, not in a Colab cell.

For an Intel Mac, Homebrew is normally under `/usr/local`. For Apple Silicon it is normally under `/opt/homebrew`.

Install the basic tools:


In [ ]:
# MAC TERMINAL COMMANDS
# Run these in Terminal, not in Google Colab.

brew update
brew install python cmake ninja pkg-config openssl@3 mosquitto git


### Mac verification

Run this in Terminal:

```bash
which python3
python3 --version
brew --prefix openssl@3
$(brew --prefix openssl@3)/bin/openssl version
mosquitto -h | head
```

**Expected result:** Python 3.x, OpenSSL 3.x, and a working Mosquitto installation.

Do not replace macOS system libraries globally. We will use Homebrew/local paths for the research environment.


## 4. Linux/Colab system packages

This cell is for Google Colab/Linux. It installs Mosquitto and Linux traffic-control utilities.



In [4]:
import subprocess, sys, os

if ENVIRONMENT == "colab":
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq mosquitto mosquitto-clients iproute2 openssl",
        shell=True,
        check=True,
    )
    print("Linux MQTT/network packages installed.")
else:
    print("Skipped: this is a Linux/Colab installation cell.")


Skipped: this is a Linux/Colab installation cell.


**Expected result:** On Colab, the installation completes without errors.

`iproute2` provides the `tc` command used for Linux network emulation. macOS does not use this tool.


## 5. Create a project directory

All generated certificates, broker configuration, raw results, and figures will stay inside one directory.

This makes the project easy to upload to GitHub while keeping large generated datasets separate if necessary.


In [5]:
from pathlib import Path

PROJECT = Path.cwd() / "pqc_mqtt_project"
CERTS = PROJECT / "certs"
RESULTS = PROJECT / "results"
FIGURES = PROJECT / "figures"
LOGS = PROJECT / "logs"

for directory in [PROJECT, CERTS, RESULTS, FIGURES, LOGS]:
    directory.mkdir(parents=True, exist_ok=True)

print(PROJECT)
print("Created:", ", ".join(str(x) for x in [CERTS, RESULTS, FIGURES, LOGS]))


/Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project
Created: /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs, /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/results, /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/figures, /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/logs


**Expected result:** Four folders are created:

- `certs/`
- `results/`
- `figures/`
- `logs/`

For GitHub, do not commit private keys or huge raw experiment files unless you intentionally decide to publish them.


## 6. Generate a local CA and broker certificate

MQTT-over-TLS needs a certificate for the broker.

For research reproducibility we create a self-signed local CA and a broker certificate. This certificate is only for the local experiment; it is **not** a production certificate.


In [6]:
import subprocess
from pathlib import Path

ca_key = CERTS / "ca.key"
ca_crt = CERTS / "ca.crt"
broker_key = CERTS / "broker.key"
broker_csr = CERTS / "broker.csr"
broker_crt = CERTS / "broker.crt"
ext_file = CERTS / "broker.ext"

subprocess.run([
    "openssl", "genrsa", "-out", str(ca_key), "2048"
], check=True)

subprocess.run([
    "openssl", "req", "-x509", "-new", "-nodes",
    "-key", str(ca_key),
    "-sha256", "-days", "365",
    "-subj", "/CN=PQC-MQTT-Test-CA",
    "-out", str(ca_crt)
], check=True)

subprocess.run([
    "openssl", "genrsa", "-out", str(broker_key), "2048"
], check=True)

subprocess.run([
    "openssl", "req", "-new",
    "-key", str(broker_key),
    "-subj", "/CN=localhost",
    "-out", str(broker_csr)
], check=True)

ext_file.write_text("subjectAltName=DNS:localhost,IP:127.0.0.1\n")

subprocess.run([
    "openssl", "x509", "-req",
    "-in", str(broker_csr),
    "-CA", str(ca_crt),
    "-CAkey", str(ca_key),
    "-CAcreateserial",
    "-out", str(broker_crt),
    "-days", "365",
    "-sha256",
    "-extfile", str(ext_file)
], check=True)

print("Certificates generated:")
for p in [ca_crt, broker_crt, broker_key]:
    print(" -", p)


Certificates generated:
 - /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/ca.crt
 - /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/broker.crt
 - /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/broker.key


Certificate request self-signature ok
subject=CN=localhost


**Expected result:** OpenSSL generates a CA certificate, broker certificate, and broker private key.

The broker certificate uses `localhost`, which matches the MQTT client connection in this experiment.


## 7. Configure Mosquitto for TLS 1.3

We use a separate configuration file rather than modifying the system Mosquitto configuration.


In [7]:
mosq_conf = PROJECT / "mosquitto.conf"

mosq_conf.write_text(f'''
listener 8883
protocol mqtt

cafile {ca_crt}
certfile {broker_crt}
keyfile {broker_key}

require_certificate false
allow_anonymous true

tls_version tlsv1.3

log_type all
'''.strip() + "\n")

print(mosq_conf.read_text())


listener 8883
protocol mqtt

cafile /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/ca.crt
certfile /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/broker.crt
keyfile /Volumes/Data Drive/Projects/Professor's projects/Frederick Sheldon/Post-Quantum-Secure-MQTT/pqc_mqtt_project/certs/broker.key

require_certificate false
allow_anonymous true

tls_version tlsv1.3

log_type all



**Expected result:** The notebook prints a Mosquitto configuration containing port `8883` and TLS 1.3.

For this research prototype, anonymous MQTT authentication is used so that authentication does not become an uncontrolled experimental variable.


## 8. Start the TLS MQTT broker

This starts Mosquitto as a background process.



In [8]:
import subprocess, os, signal, time

broker_log = LOGS / "mosquitto.log"

# Stop an old instance if this notebook was rerun.
if "BROKER_PROCESS" in globals() and BROKER_PROCESS.poll() is None:
    BROKER_PROCESS.terminate()
    time.sleep(1)

BROKER_PROCESS = subprocess.Popen(
    ["mosquitto", "-c", str(mosq_conf)],
    stdout=open(broker_log, "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(2)

print("Broker PID:", BROKER_PROCESS.pid)
print("Broker running:", BROKER_PROCESS.poll() is None)
print(broker_log.read_text()[-2000:])


FileNotFoundError: [Errno 2] No such file or directory: 'mosquitto'

**Expected result:** `Broker running: True`.

If the broker does not start, inspect `logs/mosquitto.log`. Common causes are a certificate-path problem or port `8883` already being occupied.


## 9. Test the broker with OpenSSL

Before involving Python, verify that the TLS server actually speaks TLS 1.3.


In [ ]:
result = subprocess.run(
    [
        "openssl", "s_client",
        "-connect", "127.0.0.1:8883",
        "-CAfile", str(ca_crt),
        "-tls1_3",
    ],
    input="",
    capture_output=True,
    text=True,
    timeout=15,
)

text_out = result.stdout + "\n" + result.stderr
for line in text_out.splitlines():
    if "Protocol" in line or "Cipher" in line or "Verify return code" in line:
        print(line)


**Expected result:** You should see `TLSv1.3` and a TLS 1.3 cipher such as `TLS_AES_256_GCM_SHA384`.

This verifies the TLS layer independently of MQTT.


## 10. Python MQTT/TLS client

Now create a small reusable client.

We deliberately use Paho MQTT 2.x and Python's standard TLS implementation for the classical baseline.


In [ ]:
import ssl
import time
import paho.mqtt.client as mqtt

def make_tls_client(client_id):
    client = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=client_id,
        protocol=mqtt.MQTTv5,
    )

    client.tls_set(
        ca_certs=str(ca_crt),
        tls_version=ssl.TLSVersion.TLSv1_3,
    )
    client.tls_insecure_set(False)
    return client

def mqtt_roundtrip(message=b"hello-iot"):
    client = make_tls_client(f"client-{time.time_ns()}")
    received = []

    def on_message(client, userdata, msg):
        received.append(msg.payload)

    client.on_message = on_message

    start = time.perf_counter()
    client.connect("127.0.0.1", 8883, keepalive=30)
    connect_done = time.perf_counter()

    client.loop_start()
    client.subscribe("pqc/test")
    time.sleep(0.05)

    client.publish("pqc/test", message, qos=1)
    time.sleep(0.10)

    end = time.perf_counter()
    client.loop_stop()
    client.disconnect()

    return {
        "connect_seconds": connect_done - start,
        "total_seconds": end - start,
        "received": len(received),
    }

result = mqtt_roundtrip()
print(result)


**Expected result:** A dictionary similar to:

```text
{'connect_seconds': 0.0..., 'total_seconds': 0.1..., 'received': 1}
```

The exact timing is machine-dependent. **Do not copy example numbers into the paper.** Use measurements produced by your own runs.


## 11. Measure process CPU time and memory

The paper reports CPU processing time and peak resident memory. This cell introduces a reusable measurement approach.

Because the MQTT client and TLS handshake occur inside the same Python process, `psutil.Process()` can measure the client's resource usage.


In [ ]:
import psutil
import time
import statistics

def measured_connection():
    process = psutil.Process(os.getpid())

    cpu_before = process.cpu_times()
    rss_before = process.memory_info().rss

    start = time.perf_counter()

    client = make_tls_client(f"measure-{time.time_ns()}")
    client.connect("127.0.0.1", 8883, keepalive=30)
    connect_done = time.perf_counter()

    rss_after = process.memory_info().rss
    cpu_after = process.cpu_times()

    client.disconnect()

    cpu_used = (
        (cpu_after.user - cpu_before.user)
        + (cpu_after.system - cpu_before.system)
    )

    return {
        "handshake_seconds": connect_done - start,
        "cpu_seconds": cpu_used,
        "rss_mb": rss_after / (1024 * 1024),
    }

print(measured_connection())


**Expected result:** A dictionary containing:

- `handshake_seconds`
- `cpu_seconds`
- `rss_mb`

The RSS value is the memory footprint of the Python process, so it includes Python/Paho/OpenSSL overhead. This is important when interpreting results: it is **not** the memory consumed only by ML-KEM.


## 12. Validate ML-KEM-768 independently

Before attempting to integrate ML-KEM into TLS, verify that the actual NIST-standardized ML-KEM-768 primitive works.

The Open Quantum Safe Python binding exposes KEM operations through `KeyEncapsulation`.

This test is a cryptographic validation, **not yet a TLS benchmark**.


In [ ]:
# This cell may require liboqs/liboqs-python setup first.
# If `import oqs` fails, continue to the installation cells below.

try:
    import oqs

    print("liboqs version:", oqs.oqs_version())
    enabled = oqs.get_enabled_kem_mechanisms()
    print("ML-KEM-768 enabled:", "ML-KEM-768" in enabled)
except Exception as exc:
    print("ML-KEM Python binding is not ready yet.")
    print(type(exc).__name__, exc)


**Expected result:** Ideally:

```text
ML-KEM-768 enabled: True
```

If it is `False` or import fails, do not substitute another algorithm and call it ML-KEM-768. Fix the liboqs installation first.


## 13. Install liboqs on Colab

This is a source build. It can take several minutes.

We restrict the liboqs build to ML-KEM-768 to keep the experiment smaller.


In [ ]:
if ENVIRONMENT == "colab":
    commands = [
        "rm -rf /content/liboqs /content/liboqs-python",
        "git clone --depth=1 --branch 0.14.0 https://github.com/open-quantum-safe/liboqs.git /content/liboqs",
        "cmake -S /content/liboqs -B /content/liboqs/build "
        "-GNinja -DCMAKE_BUILD_TYPE=Release "
        "-DBUILD_SHARED_LIBS=ON "
        "-DOQS_MINIMAL_BUILD='KEM_ml_kem_768'",
        "cmake --build /content/liboqs/build -j2",
        "cmake --install /content/liboqs/build",
        "git clone --depth=1 https://github.com/open-quantum-safe/liboqs-python.git /content/liboqs-python",
        "pip install -q /content/liboqs-python",
    ]

    subprocess.run(" && ".join(commands), shell=True, check=True)
    print("liboqs and liboqs-python installed.")
else:
    print("Skipped: Colab/Linux installation cell.")


**Expected result:** The build completes and `pip` installs the Python wrapper.

The version is pinned here to make the notebook more reproducible. If the OQS ecosystem changes, update the pinned versions deliberately and record the change in the GitHub README.


## 14. Install liboqs on the MacBook

Run these commands in **Mac Terminal**.



In [ ]:
# MAC TERMINAL COMMANDS

mkdir -p ~/pqc-mqtt-research
cd ~/pqc-mqtt-research

git clone --branch 0.14.0 --depth 1 https://github.com/open-quantum-safe/liboqs.git
cmake -S liboqs -B liboqs/build -GNinja   -DCMAKE_BUILD_TYPE=Release   -DBUILD_SHARED_LIBS=ON   -DOQS_MINIMAL_BUILD="KEM_ml_kem_768"

cmake --build liboqs/build -j4
sudo cmake --install liboqs/build

git clone --depth 1 https://github.com/open-quantum-safe/liboqs-python.git
cd liboqs-python
python3 -m pip install .


### Verify on the Mac

```bash
python3 -c "import oqs; print(oqs.oqs_version()); print('ML-KEM-768' in oqs.get_enabled_kem_mechanisms())"
```

Expected output should show a liboqs version and `True`.

If your Mac is Intel, the commands are the same; CMake detects the architecture.


## 15. Run an ML-KEM-768 KEM correctness test



In [ ]:
import oqs

kem_name = "ML-KEM-768"

with oqs.KeyEncapsulation(kem_name) as client_kem:
    with oqs.KeyEncapsulation(kem_name) as server_kem:
        public_key = client_kem.generate_keypair()
        ciphertext, server_secret = server_kem.encap_secret(public_key)
        client_secret = client_kem.decap_secret(ciphertext)

        print("Algorithm:", kem_name)
        print("Public key bytes:", len(public_key))
        print("Ciphertext bytes:", len(ciphertext))
        print("Shared secret bytes:", len(client_secret))
        print("Shared secrets match:", client_secret == server_secret)


**Expected result:** The final line must be:

```text
Shared secrets match: True
```

For ML-KEM-768, the standard liboqs implementation uses a 1184-byte public key, 1088-byte ciphertext, and 32-byte shared secret. These values are useful sanity checks, but the timing values must be measured on your own machine.


## 16. Benchmark ML-KEM-768 operations

This benchmark separates key generation, encapsulation, and decapsulation.

It is useful for understanding the cryptographic cost before measuring the full MQTT/TLS stack.


In [ ]:
import time
import statistics

def benchmark_mlkem(iterations=100):
    keygen_times = []
    encap_times = []
    decap_times = []

    for _ in range(iterations):
        with oqs.KeyEncapsulation(kem_name) as client:
            with oqs.KeyEncapsulation(kem_name) as server:
                t0 = time.perf_counter_ns()
                public_key = client.generate_keypair()
                t1 = time.perf_counter_ns()

                ciphertext, ss_server = server.encap_secret(public_key)
                t2 = time.perf_counter_ns()

                ss_client = client.decap_secret(ciphertext)
                t3 = time.perf_counter_ns()

                assert ss_client == ss_server

                keygen_times.append((t1-t0)/1e6)
                encap_times.append((t2-t1)/1e6)
                decap_times.append((t3-t2)/1e6)

    return {
        "keygen_mean_ms": statistics.mean(keygen_times),
        "encap_mean_ms": statistics.mean(encap_times),
        "decap_mean_ms": statistics.mean(decap_times),
        "keygen_median_ms": statistics.median(keygen_times),
        "encap_median_ms": statistics.median(encap_times),
        "decap_median_ms": statistics.median(decap_times),
    }

mlkem_benchmark = benchmark_mlkem(100)
mlkem_benchmark


**Expected result:** A dictionary containing mean and median operation times in milliseconds.

The exact numbers will differ between your MacBook and Colab. These numbers should be reported as **your measured environment**, not as universal ML-KEM performance.


## 17. Advanced stage: build the OQS OpenSSL provider

This is the step that turns the project from a standalone ML-KEM benchmark into a **TLS 1.3 PQC experiment**.

Open Quantum Safe's provider integrates post-quantum KEMs into OpenSSL 3 TLS 1.3. The provider documentation recommends OpenSSL 3.2+ for full TLS 1.3 PQ functionality.

For reproducibility, build the provider against a compatible OpenSSL/liboqs version rather than mixing arbitrary versions.


In [ ]:
# COLAB/LINUX ADVANCED BUILD
#
# This is intentionally separate from the basic notebook because it compiles
# OpenSSL/provider components and may take substantially longer.

if ENVIRONMENT == "colab":
    print("Advanced OQS provider build is intentionally manual.")
    print("See the next Markdown cell for the recommended commands.")
else:
    print("Run the provider build from Mac Terminal using the next cell.")


### OQS provider build commands

The current OQS documentation recommends:

```bash
cmake -S . -B _build
cmake --build _build
ctest --test-dir _build
cmake --install _build
```

The provider requires OpenSSL 3 and liboqs. If they are installed in non-standard locations, pass `OPENSSL_ROOT_DIR` and `liboqs_DIR`.

For your first successful experiment, I recommend using the OQS provider's tested release/version combination rather than the moving `main` branches.

After installation, verify:

```bash
openssl list -providers
openssl list -kem-algorithms -provider default -provider oqsprovider
```

You should see the OQS provider and ML-KEM-related algorithms.

**Important:** OpenSSL 3.5+ has native support for standardized ML-KEM. The OQS project notes that its provider disables duplicate standardized implementations when OpenSSL already supplies them. Record your OpenSSL version in your experiment metadata.


## 18. Verify that the TLS implementation exposes ML-KEM

Do not proceed to the TLS-level ML-KEM experiment unless the OpenSSL installation actually exposes an ML-KEM TLS group.

Use:

```bash
openssl list -tls1_3 -tls-groups
```

and inspect the output for an ML-KEM or hybrid ML-KEM group.

The exact group name can vary with OpenSSL/OQS-provider version. **Do not hard-code an old group name without checking the installed version.**


In [ ]:
# Python-side check of the OpenSSL binary.
result = subprocess.run(
    ["openssl", "version"],
    capture_output=True,
    text=True
)
print(result.stdout.strip())

result = subprocess.run(
    ["openssl", "list", "-tls1_3", "-tls-groups"],
    capture_output=True,
    text=True
)

print(result.stdout[:5000])


**Expected result:** The command should list TLS 1.3 supported groups. An ML-KEM or hybrid ML-KEM group must be present before claiming a PQC TLS experiment.


## 19. Experimental conditions

The published study used 20 network scenarios: five latency levels × four packet-loss levels, with repeated trials.

For your first reproduction, use a smaller matrix so that debugging is manageable. Once everything works, scale to the full design.


In [ ]:
LATENCIES_MS = [0, 20, 50, 100, 200]
PACKET_LOSS_PCT = [0, 1, 2, 5]

conditions = [
    {"latency_ms": latency, "loss_pct": loss}
    for latency in LATENCIES_MS
    for loss in PACKET_LOSS_PCT
]

print("Number of network conditions:", len(conditions))
print(conditions[:5])


**Expected result:**

```text
Number of network conditions: 20
```

This gives the same 5 × 4 experimental design as the paper.


## 20. Linux network impairment with tc/netem

This is the closest notebook implementation to the paper's kernel-level network emulation.

**Only run this on Linux/Colab. Do not run it on macOS.**

First identify the network interface:

```bash
ip route
```

For a local client/broker experiment, traffic may need to be routed through an appropriate interface or network namespace. Because incorrect `tc` rules can affect the whole VM, this section should be used carefully.

For a first experiment, use a dedicated Linux network namespace/container rather than modifying the primary Colab interface.


In [ ]:
if ENVIRONMENT == "colab":
    print(subprocess.run(["bash", "-lc", "ip route"], capture_output=True, text=True).stdout)
else:
    print("tc/netem is a Linux experiment. Use Colab/Linux for this section.")


### Safe recommendation

For the first GitHub version, document the `tc/netem` commands but run the full 20-condition experiment inside a controlled Linux namespace/container.

Conceptually:

```bash
sudo tc qdisc add dev <interface> root netem delay 100ms loss 2%
```

After the experiment:

```bash
sudo tc qdisc del dev <interface> root
```

Never leave an impairment rule active unintentionally.


## 21. Cross-platform fallback: application-level latency experiment

If you are working only on your MacBook, you can still demonstrate the experimental idea by introducing controlled delay in the test harness.

This is **not equivalent to kernel-level packet loss**, so label it clearly in your paper/repository.

The stronger final experiment should use Linux `tc/netem`.


In [ ]:
import time

def sleep_network_delay(latency_ms):
    time.sleep(latency_ms / 1000.0)

for latency in [0, 20, 50, 100]:
    t0 = time.perf_counter()
    sleep_network_delay(latency)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"Requested {latency:>3} ms -> measured approximately {elapsed:.2f} ms")


**Expected result:** Each measured delay should be close to the requested value, with some operating-system scheduling overhead.

Again, this demonstrates latency control but does not reproduce TCP retransmission behavior.


## 22. Create the experiment runner

The following function records one MQTT/TLS connection attempt.

The `algorithm` field is a label. For the classical baseline it can identify the TLS configuration you actually tested. For ML-KEM, only use `ML-KEM-768` after the TLS stack has successfully negotiated that group.


In [ ]:
import pandas as pd
import traceback

def run_one_trial(algorithm, latency_ms=0, loss_pct=0, trial_id=0):
    record = {
        "algorithm": algorithm,
        "latency_ms": latency_ms,
        "loss_pct": loss_pct,
        "trial": trial_id,
        "success": False,
        "handshake_ms": None,
        "cpu_ms": None,
        "rss_mb": None,
        "error": None,
    }

    try:
        # For the baseline notebook this delay is a controlled harness delay.
        # Replace this with tc/netem when running the full Linux experiment.
        sleep_network_delay(latency_ms)

        process = psutil.Process(os.getpid())
        cpu_before = process.cpu_times()
        start = time.perf_counter()

        client = make_tls_client(f"trial-{time.time_ns()}")
        client.connect("127.0.0.1", 8883, keepalive=30)

        handshake_end = time.perf_counter()
        rss = process.memory_info().rss / (1024 * 1024)
        cpu_after = process.cpu_times()

        client.disconnect()

        cpu_ms = (
            (cpu_after.user - cpu_before.user)
            + (cpu_after.system - cpu_before.system)
        ) * 1000

        record.update({
            "success": True,
            "handshake_ms": (handshake_end - start) * 1000,
            "cpu_ms": cpu_ms,
            "rss_mb": rss,
        })

    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"

    return record

print(run_one_trial("CLASSICAL-TLS13", latency_ms=20, trial_id=1))


**Expected result:** A row-like dictionary with `success=True` and timing/memory values.

If a trial fails, the error is stored rather than stopping the entire experiment. This is important because connection failures are themselves a research metric.


## 23. Run a small pilot experiment

Start with 3 trials per condition. Do not immediately run thousands of trials.


In [ ]:
pilot_records = []

for condition in conditions[:4]:
    for trial in range(3):
        pilot_records.append(
            run_one_trial(
                algorithm="CLASSICAL-TLS13",
                latency_ms=condition["latency_ms"],
                loss_pct=condition["loss_pct"],
                trial_id=trial,
            )
        )

pilot_df = pd.DataFrame(pilot_records)
pilot_df


**Expected result:** A DataFrame with one row per trial and columns for algorithm, latency, loss, success, handshake time, CPU time, memory, and error.

Check this pilot carefully before scaling up.


## 24. Save raw pilot data

Raw data should be saved before analysis so that results are reproducible.


In [ ]:
pilot_path = RESULTS / "pilot_results.csv"
pilot_df.to_csv(pilot_path, index=False)

print("Saved:", pilot_path)
print("Rows:", len(pilot_df))


**Expected result:** `pilot_results.csv` appears under `results/`.

For GitHub, you can commit a small pilot dataset and use `.gitignore` for large full-run datasets if they become too large.


## 25. Scale the experiment

The paper used 50 repetitions per algorithm/network configuration, producing 3000 observations for 3 algorithms × 20 conditions × 50 trials.

Do not run the full experiment until the pilot is stable.

For your first full run, you can start with 10 repetitions and then increase to 50.


In [ ]:
REPETITIONS = 10  # Change to 50 after the pilot is stable.

records = []

algorithms = ["CLASSICAL-TLS13"]

for algorithm in algorithms:
    for condition in conditions:
        for trial in range(REPETITIONS):
            records.append(
                run_one_trial(
                    algorithm=algorithm,
                    latency_ms=condition["latency_ms"],
                    loss_pct=condition["loss_pct"],
                    trial_id=trial,
                )
            )

df = pd.DataFrame(records)

print("Observations:", len(df))
print("Success rate:", df["success"].mean())
df.head()


**Expected result:** For one algorithm, 20 conditions × 10 repetitions = **200 observations**.

Once the PQC TLS configuration is working, set:

```python
algorithms = ["ECC-P256", "RSA-2048", "ML-KEM-768"]
```

and run 50 repetitions to approach the published 3000-observation design.

Only label the algorithms according to the **actual TLS key-establishment configuration** used by OpenSSL.


## 26. Descriptive statistics

Now summarize successful connections.


In [ ]:
successful = df[df["success"]].copy()

summary = (
    successful
    .groupby(["algorithm", "latency_ms", "loss_pct"])
    .agg(
        n=("handshake_ms", "count"),
        handshake_mean_ms=("handshake_ms", "mean"),
        handshake_median_ms=("handshake_ms", "median"),
        handshake_std_ms=("handshake_ms", "std"),
        cpu_mean_ms=("cpu_ms", "mean"),
        rss_mean_mb=("rss_mb", "mean"),
    )
    .reset_index()
)

summary.head(10)


**Expected result:** A table containing the number of successful trials and mean/median/standard deviation for handshake time, CPU, and memory.

This is the first table you can eventually export for a research report.


## 27. Connection reliability

Connection success is calculated independently from continuous performance metrics.


In [ ]:
reliability = (
    df.groupby(["algorithm", "latency_ms", "loss_pct"])
      .agg(
          attempts=("trial", "count"),
          successes=("success", "sum"),
      )
      .reset_index()
)

reliability["success_rate_pct"] = (
    reliability["successes"] / reliability["attempts"] * 100
)

reliability.head(10)


**Expected result:** A success-rate table.

This is important because a failed connection has no valid handshake-time measurement but still contributes to reliability.


## 28. Plot handshake time against network latency

This plot visualizes whether network degradation has a larger effect than algorithm selection.


In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    successful
    .groupby(["algorithm", "latency_ms"], as_index=False)["handshake_ms"]
    .mean()
)

plt.figure(figsize=(9, 5))

for algorithm in plot_df["algorithm"].unique():
    subset = plot_df[plot_df["algorithm"] == algorithm]
    plt.plot(
        subset["latency_ms"],
        subset["handshake_ms"],
        marker="o",
        label=algorithm
    )

plt.xlabel("Added latency (ms)")
plt.ylabel("Mean handshake time (ms)")
plt.title("MQTT/TLS handshake time vs network latency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


**Expected result:** A line chart showing handshake time increasing as network latency increases.

For the final paper, compare the slopes and confidence intervals across algorithms rather than relying only on visual inspection.


## 29. Plot reliability



In [ ]:
rel_plot = (
    reliability
    .groupby(["algorithm", "loss_pct"], as_index=False)["success_rate_pct"]
    .mean()
)

plt.figure(figsize=(9, 5))

for algorithm in rel_plot["algorithm"].unique():
    subset = rel_plot[rel_plot["algorithm"] == algorithm]
    plt.plot(
        subset["loss_pct"],
        subset["success_rate_pct"],
        marker="o",
        label=algorithm
    )

plt.xlabel("Packet loss (%)")
plt.ylabel("Connection success (%)")
plt.title("Connection reliability under packet loss")
plt.ylim(0, 105)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


**Expected result:** A reliability curve.

With a perfect local baseline and no actual packet loss, the curve will remain near 100%. That is expected. The interesting curve appears after running the Linux `tc/netem` experiment.


## 30. Two-way ANOVA

The paper uses two-way ANOVA to separate the effects of cryptographic algorithm and network condition.

For a first analysis, use:

- factor A: algorithm
- factor B: network condition
- dependent variable: handshake time

A more complete analysis should model latency and packet loss separately.


In [ ]:
# This analysis requires at least two algorithms and actual experimental variation.
# It is intentionally guarded so a one-algorithm pilot does not produce a misleading result.

if successful["algorithm"].nunique() >= 2:
    from statsmodels.formula.api import ols
    import statsmodels.api as sm

    anova_df = successful.copy()
    anova_df["network_condition"] = (
        anova_df["latency_ms"].astype(str)
        + "ms_"
        + anova_df["loss_pct"].astype(str)
        + "loss"
    )

    model = ols(
        "handshake_ms ~ C(algorithm) + C(network_condition) + "
        "C(algorithm):C(network_condition)",
        data=anova_df
    ).fit()

    table = sm.stats.anova_lm(model, typ=2)
    display(table)
else:
    print("ANOVA skipped: at least two actual algorithm configurations are required.")


**Expected result:** An ANOVA table with F-statistics and p-values.

Do not interpret p-values until you have enough independent repeated observations and have checked assumptions. For a serious paper, also report effect sizes and confidence intervals.


## 31. Effect-size calculation

A statistically significant result is not automatically practically important. Eta-squared helps quantify the proportion of modeled variance associated with a factor.


In [ ]:
if "table" in globals():
    effect_table = table.copy()
    effect_table["eta_sq"] = (
        effect_table["sum_sq"] / effect_table["sum_sq"].sum()
    )
    display(effect_table)
else:
    print("Run the ANOVA cell after collecting at least two actual algorithm configurations.")


**Expected result:** An additional `eta_sq` column.

A small algorithm effect combined with a large network-condition effect would support the paper's central research question: whether network conditions dominate protocol-level performance.


## 32. Export final results



In [ ]:
FINAL_RESULTS = RESULTS / "mqtt_tls_experiment_results.csv"
SUMMARY_RESULTS = RESULTS / "mqtt_tls_summary.csv"

df.to_csv(FINAL_RESULTS, index=False)
summary.to_csv(SUMMARY_RESULTS, index=False)

print("Raw results:", FINAL_RESULTS)
print("Summary:", SUMMARY_RESULTS)


**Expected result:** Two CSV files are written.

Recommended GitHub structure:

```text
pqc-mqtt-research/
├── notebooks/
│   └── PQC_MQTT_TLS_Research.ipynb
├── results/
│   ├── README.md
│   └── sample_results.csv
├── figures/
├── scripts/
├── requirements.txt
├── README.md
└── .gitignore
```


## 33. Research interpretation template

After completing the full experiment, answer these questions:

### RQ1
Does ML-KEM-768 introduce statistically significant handshake overhead compared with the classical baselines?

### RQ2
Does network latency/packet loss explain more variance than cryptographic algorithm choice?

### RQ3
Does ML-KEM-768 maintain comparable connection reliability?

### RQ4 — your proposed extension
Does ML-KEM-768 remain practical when computational resources are constrained?

The first three reproduce/extend the paper. **RQ4 can become your independent contribution.**


## 34. Important limitation: MacBook vs Linux

For the final research version:

### Run on MacBook

Use the Mac for:

- development
- Python code
- ML-KEM primitive benchmarks
- MQTT broker/client debugging
- TLS/OpenSSL development
- plotting and analysis

### Run on Google Colab/Linux

Use Linux for:

- `tc/netem`
- controlled packet loss
- controlled network latency
- full 20-condition network experiment

### Later hardware extension

For your proposed research contribution, move the client to a Raspberry Pi or another constrained Linux device and repeat the same experiment.

That creates a much stronger progression:

```text
Desktop SITL
     ↓
Colab/Linux network emulation
     ↓
Resource-constrained device
     ↓
ML-KEM-768 vs RSA-2048 vs ECC-P256
     ↓
Statistical analysis
```


## 35. Reproducibility checklist

Before publishing the GitHub repository, record:

- Python version
- OS and architecture
- OpenSSL version
- Mosquitto version
- Paho-MQTT version
- liboqs version
- OQS provider version
- exact TLS group names
- latency levels
- packet-loss levels
- number of repetitions
- random seeds where applicable
- raw CSV results
- analysis notebook
- generated figures

Do not report results without recording the software versions that generated them.


## 36. What counts as a successful reproduction?

The project is successful when you can demonstrate all of the following:

1. MQTT communication works.
2. MQTT is protected with TLS 1.3.
3. The classical baseline is measured automatically.
4. ML-KEM-768 works independently through liboqs.
5. OpenSSL actually exposes ML-KEM/hybrid TLS support.
6. The MQTT client successfully negotiates the intended PQ TLS configuration.
7. Latency and packet loss can be controlled in Linux.
8. Each trial records handshake time, CPU, RSS, bytes/reliability where implemented.
9. The experiment can be repeated automatically.
10. Results are exported as CSV and figures.
11. Statistical analysis distinguishes algorithm effects from network effects.

Only after item **6** should the project be described as an **ML-KEM-over-TLS MQTT experiment** rather than an ML-KEM cryptographic benchmark.


## References

1. Almutairi, M. G., & Sheldon, F. T. (2026). *Network Conditions Dominate MQTT-over-TLS 1.3 Performance: A Comparative Evaluation of ML-KEM-768, RSA-2048, and ECC-P256*. Electronics, 15(17), 3914.

2. Almutairi, M. G., & Sheldon, F. T. (2026). *Methodology and Architecture for Benchmarking End-to-End PQC Protocol Resilience in an IoT Context*. Internet of Things, 7(1), 17.

3. Open Quantum Safe. `liboqs` and `oqs-provider` documentation.

This notebook is an independent implementation/reproduction exercise. Clearly distinguish your own measurements and extensions from results reported in the published papers.
